# Enformer Regulatory Activity Prediction

![Enformer Regulatory Activity Prediction](https://proto-bio.github.io/proto-assets/images/tool/enformer/hero.png)

Enformer is a deep learning model for predicting gene regulatory activity directly from DNA sequence, published by Avsec et al. (2021). It processes 196,608 bp input windows and outputs binned predictions across 896 spatial bins for thousands of regulatory tracks covering chromatin accessibility, histone modifications, and gene expression. The model supports both human and mouse output heads, and track selection lets you extract only the signals relevant to your analysis.

This notebook demonstrates how to predict regulatory activity from a DNA sequence and how to perform a variant-effect comparison by running reference and alternate allele windows through the model and computing log2 fold-change.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("enformer")
display_overview("enformer")
display_docs_section("enformer", "Background")

## Available tools

In [ ]:
display_available_tools("enformer")

### `run_enformer`

Enformer predicts regulatory activity from a 196,608 bp DNA input window, returning a signal matrix of shape [896, num_tracks] where each of the 896 spatial bins covers a 128 bp window. The `output_tracks` parameter selects which of the thousands of regulatory tracks to extract, and the `species` parameter switches between human and mouse output heads. A common use is variant-effect analysis: running the same input window with reference and alternate alleles and comparing the resulting prediction matrices to identify regulatory impact.

In [ ]:
import numpy as np
from proto_tools import ENFORMER_CONTEXT, EnformerConfig, EnformerInput, run_enformer

In [ ]:
# Display input docs
display_api_reference("enformer", "input", "run_enformer")

from proto_tools import NCBIEfetchConfig, NCBIEfetchInput, run_ncbi_efetch

# Fetch a real human genomic window on chr11 centred on the HBB locus, exactly the
# length Enformer requires, instead of a synthetic repeat.
_center = 5227430  # HBB midpoint, GRCh38 chr11 (NC_000011.10)
_fetch = run_ncbi_efetch(
    NCBIEfetchInput(
        db="nuccore", identifier="NC_000011.10", return_format="fasta",
        seq_start=_center - ENFORMER_CONTEXT // 2, seq_stop=_center + ENFORMER_CONTEXT // 2,
    ),
    NCBIEfetchConfig(),
)
sequence = _fetch.fasta_records[0].sequence.upper()[:ENFORMER_CONTEXT]
inputs = EnformerInput(sequences=[sequence])

In [ ]:
# Display config docs
display_api_reference("enformer", "config", "run_enformer")

config = EnformerConfig(
    output_tracks=[0, 1, 2],
    species="human",
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

In [ ]:
# Run the prediction function
pred_result = run_enformer(inputs, config)

In [ ]:
display_api_reference("enformer", "output", "run_enformer")

print(f"Output bins: {len(pred_result.results[0].prediction)}")
print(f"Tracks per bin: {len(pred_result.results[0].prediction[0])}")

#### Variant-Effect Comparison

A common use of Enformer is comparing predictions between a reference and an alternate allele to identify regulatory variants. We introduce a single-nucleotide change at the center of the input window, run both sequences through the model, and compute log2 fold-change across all bins and tracks. Large absolute fold-change values indicate positions and tracks where the variant has a meaningful effect on predicted regulatory activity.

In [ ]:
# Introduce a single-nucleotide substitution at the center of the input window
center = ENFORMER_CONTEXT // 2
ref_sequence = sequence
alt_base = "A" if ref_sequence[center] != "A" else "C"
alt_sequence = ref_sequence[:center] + alt_base + ref_sequence[center + 1:]

ref_result = run_enformer(EnformerInput(sequences=[ref_sequence]), config)
alt_result = run_enformer(EnformerInput(sequences=[alt_sequence]), config)

ref_pred = np.array(ref_result.results[0].prediction)
alt_pred = np.array(alt_result.results[0].prediction)
log2_fc = np.log2((alt_pred + 1e-6) / (ref_pred + 1e-6))
print(f"Max |log2 FC|: {np.abs(log2_fc).max():.4f}")

### `run_enformer_gradient`

The gradient tool turns Enformer's track predictions into a differentiable objective. Each term names an output track, a span of 128 bp output bins, a direction, and a weight; the terms are signed so the summed objective is always minimized, and the gradient comes back with respect to a relaxed distribution over the bases being designed.

Enformer reads a fixed 196,608 bp context, but a design problem usually varies only a small part of it. A candidate therefore bundles the **whole** context sequence with the 1-indexed `positions` whose bases are relaxed and a `(len(positions), 4)` logits matrix aligned to them — so the gradient covers only the designed positions while the surrounding context still reaches the model.

Where the variant-effect comparison above asks *what does this substitution do*, the gradient answers *which substitution would help* — it is the same model read as a design objective rather than a predictor.

In [ ]:
from proto_tools import (
    ENFORMER_CONTEXT, EnformerGradientCandidate, EnformerGradientConfig,
    EnformerGradientInput, EnformerGradientLossTerm, run_enformer_gradient,
)

display_api_reference("enformer", "input", "run_enformer_gradient")

# Design a 24 bp stretch at the centre of the same window used above, where
# Enformer's output bins are densest. Positions are 1-indexed and inclusive.
grad_positions = [ENFORMER_CONTEXT // 2 + offset for offset in range(24)]
grad_candidate = EnformerGradientCandidate(
    sequence=sequence,
    positions=grad_positions,
    # Start from a flat distribution over A, C, G, T at each designed position.
    logits=[[0.25, 0.25, 0.25, 0.25]] * len(grad_positions),
)


In [ ]:
display_api_reference("enformer", "config", "run_enformer_gradient")

# Push track 0 up over the output bins covering the designed window.
grad_config = EnformerGradientConfig(
    loss_terms=[EnformerGradientLossTerm(track=0, bin_start=440, bin_end=456, direction="max")],
)

grad_output = run_enformer_gradient(EnformerGradientInput(inputs=grad_candidate), grad_config)


In [ ]:
display_api_reference("enformer", "output", "run_enformer_gradient")

# One result per candidate; this call passed one.
grad_result = grad_output.results[0]
print(f"Weighted objective: {grad_result.loss:.4f}")
print(f"Designed positions: {grad_result.metrics['designed_positions']}")
print(f"Gradient shape:     ({len(grad_result.gradient)}, {len(grad_result.gradient[0])})")
for grad_term in grad_result.metrics["loss_terms"]:
    print(f"  track {grad_term['track']} bins {grad_term['bin_start']}-{grad_term['bin_end']}: "
          f"mean activity {grad_term['mean_activity']:.4f} ({grad_term['direction']})")
display_api_reference("enformer", "input", "run_enformer_gradient")
display_api_reference("enformer", "config", "run_enformer_gradient")
display_api_reference("enformer", "output", "run_enformer_gradient")


## Export Results

Prediction outputs can be saved to standard file formats for downstream analysis or integration with other computational pipelines. Supported formats include JSON and CSV.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

pred_result.export(name="enformer_prediction", export_path=output_dir, file_format="json")
print(f"Prediction exported to {output_dir}/enformer_prediction.json")

ref_result.export(name="enformer_ref", export_path=output_dir, file_format="csv")
print(f"Reference summary exported to {output_dir}/enformer_ref.csv")